# MGMT298D: Science and Strategy of AI## Week 7: Word Embeddings & Transformers### UCLA Anderson School of Management

## PART A: Word Embeddings with GloVe

### 1. Imports

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.decomposition import PCAfrom sklearn.metrics.pairwise import cosine_similarityimport warningswarnings.filterwarnings('ignore')sns.set_style('whitegrid')%matplotlib inline

### 2. Load GloVe EmbeddingsGloVe (Global Vectors) learns word representations from large text corpora.Words that appear in similar contexts end up with similar vectors.We use the `gensim` library to load pre-trained 50-dimensional GloVe vectors.

In [ ]:
# Download pre-trained GloVe embeddings (50-dimensional, trained on Wikipedia)import gensim.downloader as apiprint("Downloading GloVe embeddings (this may take a minute)...")glove = api.load('glove-wiki-gigaword-50')print(f"Vocabulary size: {len(glove):,} words")print(f"Embedding dimension: {glove.vector_size}")print(f"\nExample — vector for 'king' (first 10 dimensions):")print(glove['king'][:10].round(4))

### 3. Word SimilarityCosine similarity measures how close two word vectors are in the embedding space.Words used in similar contexts should have high similarity.

In [ ]:
# Compute cosine similarity between word pairstest_pairs = [    ('king', 'queen'),    ('king', 'banana'),    ('happy', 'joyful'),    ('happy', 'sad'),    ('dog', 'cat'),    ('dog', 'computer'),    ('paris', 'france'),    ('tokyo', 'japan'),]print("Word Pair Similarities:\n")for w1, w2 in test_pairs:    sim = glove.similarity(w1, w2)    print(f"  {w1:10s} ~ {w2:10s}: {sim:.4f}")

In [ ]:
# Find the most similar words to a given wordquery_words = ['king', 'computer', 'happy', 'ocean']print("Most Similar Words:\n")for word in query_words:    similar = glove.most_similar(word, topn=5)    neighbors = ', '.join([f"{w} ({s:.2f})" for w, s in similar])    print(f"  {word:10s} → {neighbors}")

### 4. Word AnalogiesWord embeddings capture relationships: king - man + woman ≈ queen.This arithmetic in vector space reveals learned semantic structure.

In [ ]:
# Solve word analogies: a is to b as c is to ?analogies = [    ('king',   'man',    'woman'),    # royalty gender    ('paris',  'france', 'japan'),    # capital-country    ('bigger', 'big',    'small'),    # comparative    ('walking','walk',   'swim'),     # verb tense]print("Word Analogies: a - b + c = ?\n")for a, b, c in analogies:    # result ≈ a - b + c    results = glove.most_similar(positive=[a, c], negative=[b], topn=3)    answers = ', '.join([f"{w} ({s:.2f})" for w, s in results])    print(f"  {a} - {b} + {c}")    print(f"    → {answers}\n")

### 5. Embedding Visualization (PCA)We project high-dimensional word vectors down to 2D using PCA.Words in the same semantic group should cluster together.

In [ ]:
# Select words from distinct semantic groupsword_groups = {    'royalty':  ['king', 'queen', 'prince', 'princess', 'throne'],    'family':   ['man', 'woman', 'boy', 'girl', 'father', 'mother'],    'animals':  ['dog', 'cat', 'lion', 'tiger', 'horse', 'bear'],    'food':     ['apple', 'bread', 'cheese', 'rice', 'chicken', 'milk'],    'places':   ['paris', 'london', 'tokyo', 'berlin', 'rome', 'madrid'],}# Collect embeddings and labelswords, vectors, groups = [], [], []for group, word_list in word_groups.items():    for w in word_list:        if w in glove:            words.append(w)            vectors.append(glove[w])            groups.append(group)vectors = np.array(vectors)# PCA to 2Dpca = PCA(n_components=2)coords = pca.fit_transform(vectors)# Plotfig, ax = plt.subplots(figsize=(12, 8))palette = sns.color_palette('husl', len(word_groups))color_map = {g: palette[i] for i, g in enumerate(word_groups)}for i, (word, group) in enumerate(zip(words, groups)):    ax.scatter(coords[i, 0], coords[i, 1], color=color_map[group], s=120, alpha=0.8, zorder=2)    ax.annotate(word, (coords[i, 0], coords[i, 1]),                xytext=(6, 6), textcoords='offset points', fontsize=10, fontweight='bold')ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)ax.set_title('GloVe Word Embeddings — PCA Projection', fontsize=14, fontweight='bold')ax.grid(True, alpha=0.3)from matplotlib.patches import Patchlegend_elements = [Patch(facecolor=color_map[g], label=g) for g in word_groups]ax.legend(handles=legend_elements, loc='best', framealpha=0.9, fontsize=10)plt.tight_layout()plt.show()

## PART B: Custom Transformer — AG News Classification

### 6. Load AG News DatasetAG News contains 120k news articles in 4 categories: World, Sports, Business, and Sci/Tech.We'll train a transformer from scratch to classify these articles.

In [ ]:
import tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersfrom tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling1Dfrom datasets import load_datasetfrom sklearn.metrics import confusion_matrix, classification_report, accuracy_score# Load AG News from HuggingFacedataset = load_dataset('ag_news')CLASS_NAMES = ['World', 'Sports', 'Business', 'Sci/Tech']print(f"Training samples: {len(dataset['train']):,}")print(f"Test samples:     {len(dataset['test']):,}")print(f"Classes:          {CLASS_NAMES}")print(f"\nSample articles:")for i in range(3):    label = CLASS_NAMES[dataset['train'][i]['label']]    text = dataset['train'][i]['text'][:100]    print(f"  [{label}] {text}...")

### 7. Prepare DataWe tokenize the text, convert to padded sequences, and take a training subset to keep training fast.

In [ ]:
# Use a subset for faster trainingTRAIN_SIZE = 20000TEST_SIZE = 4000VOCAB_SIZE = 15000MAX_LEN = 200# Extract text and labelstrain_texts = dataset['train']['text'][:TRAIN_SIZE]train_labels = np.array(dataset['train']['label'][:TRAIN_SIZE])test_texts = dataset['test']['text'][:TEST_SIZE]test_labels = np.array(dataset['test']['label'][:TEST_SIZE])# Create Keras TextVectorization layer for tokenizationvectorizer = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LEN)vectorizer.adapt(train_texts)# Vectorize textx_train = vectorizer(np.array(train_texts)).numpy()x_test = vectorizer(np.array(test_texts)).numpy()y_train = train_labelsy_test = test_labelsprint(f"Training set: {x_train.shape}")print(f"Test set:     {x_test.shape}")print(f"Vocabulary:   {VOCAB_SIZE:,} tokens")print(f"Sequence len: {MAX_LEN}")

### 8. Build Transformer BlockWe define a TransformerBlock with multi-head self-attention and a feed-forward network,plus residual connections and layer normalization — the core building block of all transformer models.

In [ ]:
# Custom Transformer Blockclass TransformerBlock(layers.Layer):    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):        super().__init__()        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)        self.ffn = keras.Sequential([            Dense(ff_dim, activation='relu'),            Dense(embed_dim),        ])        self.norm1 = layers.LayerNormalization(epsilon=1e-6)        self.norm2 = layers.LayerNormalization(epsilon=1e-6)        self.drop1 = Dropout(rate)        self.drop2 = Dropout(rate)    def call(self, inputs, training):        attn = self.att(inputs, inputs)        attn = self.drop1(attn, training=training)        out1 = self.norm1(inputs + attn)       # Residual + LayerNorm        ffn = self.ffn(out1)        ffn = self.drop2(ffn, training=training)        return self.norm2(out1 + ffn)          # Residual + LayerNorm# Token + Positional Embeddingclass TokenAndPositionEmbedding(layers.Layer):    def __init__(self, maxlen, vocab_size, embed_dim):        super().__init__()        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)    def call(self, x):        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)        return self.token_emb(x) + self.pos_emb(positions)print("Defined: TransformerBlock (MultiHeadAttention + FFN + Residuals)")print("Defined: TokenAndPositionEmbedding (Token + Positional)")

### 9. Build & Train Classifier

In [ ]:
# Build transformer classifier with configurable depthEMBED_DIM = 64NUM_HEADS = 2FF_DIM = 64def build_transformer_classifier(num_blocks=1):    inputs = layers.Input(shape=(MAX_LEN,))    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)        for _ in range(num_blocks):        x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM, rate=0.1)(x)        x = GlobalAveragePooling1D()(x)    x = Dropout(0.1)(x)    x = Dense(32, activation='relu')(x)    outputs = Dense(4, activation='softmax')(x)    return keras.Model(inputs=inputs, outputs=outputs)# --- Train 1-block model ---model_1block = build_transformer_classifier(num_blocks=1)model_1block.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])print("Training 1-block Transformer...")history_1b = model_1block.fit(x_train, y_train, batch_size=64, epochs=10,                              validation_split=0.1, verbose=0)acc_1b = model_1block.evaluate(x_test, y_test, verbose=0)[1]print(f"1-Block Test Accuracy: {acc_1b:.4f}  |  Parameters: {model_1block.count_params():,}")# --- Train 3-block model ---model_3block = build_transformer_classifier(num_blocks=3)model_3block.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])print("\nTraining 3-block Transformer...")history_3b = model_3block.fit(x_train, y_train, batch_size=64, epochs=10,                              validation_split=0.1, verbose=0)acc_3b = model_3block.evaluate(x_test, y_test, verbose=0)[1]print(f"3-Block Test Accuracy: {acc_3b:.4f}  |  Parameters: {model_3block.count_params():,}")

### 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))for hist, label in [(history_1b, '1-Block'), (history_3b, '3-Block')]:    axes[0].plot(hist.history['accuracy'], label=f'{label} Train', linewidth=2)    axes[0].plot(hist.history['val_accuracy'], label=f'{label} Val', linewidth=2, linestyle='--')axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Accuracy')axes[0].set_title('Training & Validation Accuracy')axes[0].legend()axes[0].grid(True, alpha=0.3)for hist, label in [(history_1b, '1-Block'), (history_3b, '3-Block')]:    axes[1].plot(hist.history['loss'], label=f'{label} Train', linewidth=2)    axes[1].plot(hist.history['val_loss'], label=f'{label} Val', linewidth=2, linestyle='--')axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Loss')axes[1].set_title('Training & Validation Loss')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

### 11. Confusion Matrix — Best Custom Model

In [ ]:
# Use the better-performing custom modelbest_custom = model_3block if acc_3b >= acc_1b else model_1blockbest_custom_name = '3-Block' if acc_3b >= acc_1b else '1-Block'best_custom_acc = max(acc_1b, acc_3b)y_pred_custom = np.argmax(best_custom.predict(x_test, verbose=0), axis=1)cm = confusion_matrix(y_test, y_pred_custom)plt.figure(figsize=(8, 6))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,            cbar_kws={'label': 'Count'})plt.xlabel('Predicted')plt.ylabel('Actual')plt.title(f'Confusion Matrix — {best_custom_name} Custom Transformer')plt.tight_layout()plt.show()print(classification_report(y_test, y_pred_custom, target_names=CLASS_NAMES))

## PART C: Pre-trained Transformer — Transfer Learning

### 12. Zero-Shot ClassificationA pre-trained model can classify text *without ever seeing AG News*.We simply describe the categories in natural language and let the model match.

In [ ]:
from transformers import pipeline# Load zero-shot classification pipeline (downloads model on first run)print("Loading zero-shot classifier (facebook/bart-large-mnli)...")zs_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",                          device=0 if tf.config.list_physical_devices('GPU') else -1)# Classify a sample of test articlesZS_SAMPLE = 200sample_texts = dataset['test']['text'][:ZS_SAMPLE]sample_labels = dataset['test']['label'][:ZS_SAMPLE]candidate_labels = ['world news', 'sports', 'business', 'science and technology']print(f"Running zero-shot on {ZS_SAMPLE} test samples...\n")zs_preds = []for text in sample_texts:    result = zs_classifier(text, candidate_labels)    pred_idx = candidate_labels.index(result['labels'][0])    zs_preds.append(pred_idx)acc_zs = accuracy_score(sample_labels, zs_preds)print(f"Zero-Shot Accuracy ({ZS_SAMPLE} samples): {acc_zs:.4f}")print("\nReminder: this model has NEVER been trained on AG News!")

In [ ]:
# Show a few zero-shot predictionsprint("Sample Zero-Shot Predictions:\n")for i in range(5):    true = CLASS_NAMES[sample_labels[i]]    pred = CLASS_NAMES[zs_preds[i]]    match = '✓' if sample_labels[i] == zs_preds[i] else '✗'    print(f"  {match} True: {true:10s} | Pred: {pred:10s} | {sample_texts[i][:80]}...")

### 13. Fine-Tune DistilBERTNow we fine-tune a pre-trained DistilBERT model on AG News.This combines the language knowledge from pre-training with task-specific learning.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,                          TrainingArguments, Trainer)from datasets import Datasetimport torch# Prepare HuggingFace datasets for fine-tuningtokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')# Use same subset sizes as Part B for fair comparisontrain_ds = Dataset.from_dict({    'text': dataset['train']['text'][:TRAIN_SIZE],    'label': dataset['train']['label'][:TRAIN_SIZE]})test_ds = Dataset.from_dict({    'text': dataset['test']['text'][:TEST_SIZE],    'label': dataset['test']['label'][:TEST_SIZE]})def tokenize_fn(batch):    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)train_ds = train_ds.map(tokenize_fn, batched=True)test_ds = test_ds.map(tokenize_fn, batched=True)print(f"Tokenized train: {len(train_ds):,} samples")print(f"Tokenized test:  {len(test_ds):,} samples")

In [ ]:
# Load pre-trained DistilBERT and add a classification headmodel_ft = AutoModelForSequenceClassification.from_pretrained(    'distilbert-base-uncased', num_labels=4)# Training configurationtraining_args = TrainingArguments(    output_dir='./results',    num_train_epochs=3,    per_device_train_batch_size=32,    per_device_eval_batch_size=64,    eval_strategy='epoch',    logging_strategy='epoch',    save_strategy='no',    learning_rate=2e-5,    weight_decay=0.01,    report_to='none',)trainer = Trainer(    model=model_ft,    args=training_args,    train_dataset=train_ds,    eval_dataset=test_ds,)print("Fine-tuning DistilBERT (3 epochs)...")trainer.train()

### 14. Evaluate Fine-Tuned Model

In [ ]:
# Generate predictions on test setft_predictions = trainer.predict(test_ds)y_pred_ft = np.argmax(ft_predictions.predictions, axis=1)acc_ft = accuracy_score(test_ds['label'], y_pred_ft)print(f"Fine-Tuned DistilBERT Test Accuracy: {acc_ft:.4f}")print(f"Parameters: {model_ft.num_parameters():,}\n")# Confusion matrixcm_ft = confusion_matrix(test_ds['label'], y_pred_ft)plt.figure(figsize=(8, 6))sns.heatmap(cm_ft, annot=True, fmt='d', cmap='Greens',            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,            cbar_kws={'label': 'Count'})plt.xlabel('Predicted')plt.ylabel('Actual')plt.title('Confusion Matrix — Fine-Tuned DistilBERT')plt.tight_layout()plt.show()print(classification_report(test_ds['label'], y_pred_ft, target_names=CLASS_NAMES))

### 15. Final Model Comparison

In [ ]:
# Compare all modelsmodel_names = ['1-Block\nTransformer', '3-Block\nTransformer',               f'Zero-Shot\n(n={ZS_SAMPLE})', 'Fine-Tuned\nDistilBERT']accs = [acc_1b, acc_3b, acc_zs, acc_ft]colors = ['#1f77b4', '#ff7f0e', '#9467bd', '#2ca02c']fig, ax = plt.subplots(figsize=(10, 5))bars = ax.bar(model_names, accs, color=colors, alpha=0.85, edgecolor='black', linewidth=0.8)for bar, acc in zip(bars, accs):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,            f'{acc:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')ax.set_ylabel('Test Accuracy', fontsize=12)ax.set_title('AG News Classification — Model Comparison', fontsize=14, fontweight='bold')ax.set_ylim([0, 1.05])ax.grid(True, alpha=0.3, axis='y')# Add divider between custom and pre-trainedax.axvline(x=1.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.6)ax.text(0.5, 1.02, 'Trained from Scratch', ha='center', fontsize=10,        fontstyle='italic', color='gray', transform=ax.get_xaxis_transform())ax.text(2.5, 1.02, 'Pre-trained', ha='center', fontsize=10,        fontstyle='italic', color='gray', transform=ax.get_xaxis_transform())plt.tight_layout()plt.show()print("\nKey Takeaway:")print(f"  From-scratch best:  {max(acc_1b, acc_3b):.4f}")print(f"  Fine-tuned BERT:    {acc_ft:.4f}")print(f"  Improvement:        {(acc_ft - max(acc_1b, acc_3b))*100:+.1f} percentage points")print(f"\n  Pre-training on massive text corpora gives DistilBERT a huge advantage,")print(f"  even with the same amount of task-specific training data.")